In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
import pandas as pd

df = pd.read_hdf(
    "metr-la.h5",
    key="df"
)

traffic = df.values

print(traffic.shape)

(34272, 207)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(34260, 12, 207)
(34260, 207)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [9]:
class ClusterHypergraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_clusters,
        channels
    ):

        super().__init__()

        self.num_nodes = num_nodes
        self.num_clusters = num_clusters

        self.cluster_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                num_clusters
            )
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        H = torch.softmax(
            self.cluster_embeddings,
            dim=1
        )

        hyper_adj = H @ H.T

        hyper_adj = hyper_adj / (
            hyper_adj.sum(
                dim=1,
                keepdim=True
            )
            + 1e-6
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            hyper_adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [10]:
class CAHSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(
            1,
            32
        )

        self.hypergraph = ClusterHypergraphConv(
            num_nodes=207,
            num_clusters=16,
            channels=32
        )

        self.temp2 = TemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.hypergraph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [11]:
model = CAHSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 207])
torch.Size([64, 207])


In [12]:
model = CAHSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.075968
Epoch 2/30 Loss: 0.038538
Epoch 3/30 Loss: 0.032864
Epoch 4/30 Loss: 0.028129
Epoch 5/30 Loss: 0.025315
Epoch 6/30 Loss: 0.023353
Epoch 7/30 Loss: 0.022032
Epoch 8/30 Loss: 0.020948
Epoch 9/30 Loss: 0.020381
Epoch 10/30 Loss: 0.020002
Epoch 11/30 Loss: 0.019626
Epoch 12/30 Loss: 0.019385
Epoch 13/30 Loss: 0.019144
Epoch 14/30 Loss: 0.018990
Epoch 15/30 Loss: 0.018853
Epoch 16/30 Loss: 0.018716
Epoch 17/30 Loss: 0.018554
Epoch 18/30 Loss: 0.018428
Epoch 19/30 Loss: 0.018236
Epoch 20/30 Loss: 0.018179
Epoch 21/30 Loss: 0.018096
Epoch 22/30 Loss: 0.018021
Epoch 23/30 Loss: 0.017934
Epoch 24/30 Loss: 0.017868
Epoch 25/30 Loss: 0.017810
Epoch 26/30 Loss: 0.017755
Epoch 27/30 Loss: 0.017694
Epoch 28/30 Loss: 0.017613
Epoch 29/30 Loss: 0.017548
Epoch 30/30 Loss: 0.017525


In [13]:
train_time = time.time() - train_start
print("Time Taken:", train_time)

Time Taken: 3507.7413148880005


In [14]:
torch.save(
    model.state_dict(),
    "CAH-STGCN-PEMS-BAY.pth"
)

In [15]:
test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 11.936357259750366
MAE: 0.08149467
RMSE: 0.15776753


In [16]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)

print("MAPE:", mape)
print("R2:", r2)

MAPE: 1900005.078125
R2: 0.7649783080265689


In [17]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 7633
